# 第1回：Tiny ViT（MNIST）学習

**到達目標**
- MNIST上で Tiny ViT（軽量ViT）を学習し，精度を評価できる

> 計算を軽くするため、エポック数は少なめの設定です．学習時間に応じて変更してください．

In [1]:
import sys, platform, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Platform:', platform.platform())

Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
PyTorch: 2.11.0
CUDA available: False
Platform: macOS-15.5-arm64-arm-64bit


## 1. ライブラリのインポート & 乱数シード固定

In [2]:
import os, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

def set_seed(seed=2025):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(2025)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## 2. データセット（MNIST）
- 入力は 1ch（28×28）．ViTの都合上，`Patch size=7` で 4×4=16 トークンに分割します．
- 前処理は `[0,1]` 正規化のみ（単純化）．

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

100.0%
100.0%
100.0%
100.0%


## 3. Tiny ViT（MNIST向け）実装

In [4]:
class PatchEmbed(nn.Module):
    def __init__(self, in_ch=1, embed_dim=64, patch_size=7):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)                      # [B,E,4,4]
        x = x.flatten(2).transpose(1, 2)      # [B,16,E]
        return x

class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Attention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1,2).reshape(B, N, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = MLP(dim, hidden)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class TinyViT_MNIST(nn.Module):
    def __init__(self, embed_dim=64, depth=2, num_heads=4, num_classes=10):
        super().__init__()
        self.patch = PatchEmbed(1, embed_dim, patch_size=7)
        self.pos = nn.Parameter(torch.zeros(1, 16, embed_dim))
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads=num_heads, mlp_ratio=2.0) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos, std=0.02)
    def forward(self, x):
        x = self.patch(x) + self.pos
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        logits = self.head(x)
        return logits

## 4. 学習ループ（短時間版）

In [5]:
def train(model, loader, optimizer, epoch, device):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for (x, y) in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for (x, y) in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return loss_sum/total, correct/total

model = TinyViT_MNIST(embed_dim=64, depth=2, num_heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
EPOCHS = 5
for ep in range(1, EPOCHS+1):
    tr_loss, tr_acc = train(model, train_loader, optimizer, ep, device)
    te_loss, te_acc = evaluate(model, test_loader, device)
    print(f'Epoch {ep:02d} | Train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | Test loss {te_loss:.4f} acc {te_acc*100:.2f}%')

/opt/homebrew/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 